In [1]:
!pip install wandb

In [2]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')  # Optional: Download WordNet's extended multilingual data

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


True

In [3]:
!pip install rouge-score

  Preparing metadata (setup.py) ... - done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ab24e8855063d276d2b8e14b28b141b1655af12e46f4128c23614ffdd35c6feb
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [4]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration, AdamW, get_linear_schedule_with_warmup
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import wandb
from rouge_score import rouge_scorer
# from dotenv import load_dotenv
import os
from tabulate import tabulate
from nltk.translate.bleu_score import corpus_bleu
import sympy as sp
# load_dotenv()

In [5]:
df = pd.read_csv('/kaggle/input/rag-datasetcontext/RAG_QA_dataset.csv')

# If 'Unnamed: 0' column still exists, you can drop it
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 80% -> Training Data, 20% -> Testing Data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 90% -> Training Data, 10% -> Validation Data
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

In [6]:
test_df.to_csv("test_df.csv")

In [7]:
df.head()

,Question,Object,Answer,Context
0,what is a yali ?,yali,a yali is a mythical creature found predominan...,Yali is a mythical creature found predominantl...
1,what animals is a yali typically composed of ?,yali,a yali is typically depicted as a composite of...,Yali is a mythical creature found predominantl...
2,what attributes does a yali symbolize ?,yali,"a yali symbolizes attributes like strength, pr...",Yali is a mythical creature found predominantl...
3,why are yalis considered unique ?,yali,yalis are unique because they do not adhere to...,Yali is a mythical creature found predominantl...
4,which cultures have similar mythical creatures...,yali,cultures that have similar mythical creatures ...,It shares similarities with other mythical cre...


In [8]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

def calculate_max_length(column_name):
    df[column_name] = df[column_name].astype(str)
    return df[column_name].apply(lambda x: len(tokenizer.tokenize(x))).max()

max_length_question = calculate_max_length('Question')
max_length_object = calculate_max_length('Object')
max_length_answer = calculate_max_length('Answer')
max_length_context = calculate_max_length('Context')

print(f"Maximum token length in the 'Question' column: {max_length_question + max_length_object + max_length_context}")
print(f"Maximum token length in the 'Answer' column: {max_length_answer}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Maximum token length in the 'Question' column: 184
Maximum token length in the 'Answer' column: 62


In [9]:
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

In [10]:
class Dataset(Dataset):
    '''For Loading the datasetS! '''
    def __init__(self, data, question_max_length=200, answer_max_length=80):
        self.data = data
        self.tokenizer = tokenizer
        self.question_max_length = question_max_length
        self.answer_max_length = answer_max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        question = self.data.iloc[idx, 0]
        object_name = self.data.iloc[idx, 1]
        answer = self.data.iloc[idx, 2]
        context = self.data.iloc[idx, 3]
        combined = str(object_name) + ' ' + str(question) + ''  + str(context)

        inputs = self.tokenizer(
            combined,
            max_length=self.question_max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            answer,
            max_length=self.answer_max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        input_ids = inputs.input_ids.squeeze()
        attention_mask = inputs.attention_mask.squeeze()
        target_ids = targets.input_ids.squeeze()

        return {

            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': target_ids
        }

In [11]:
model

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
    

In [12]:
train_dataset = Dataset(train_df)
val_dataset = Dataset(val_df)
test_dataset = Dataset(test_df)

In [13]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 5.2 MB/s eta 0:00:00


In [14]:
from rouge_score import rouge_scorer
from sklearn.metrics.pairwise import cosine_similarity
# from nltk.translate.meteor_score import meteor_score
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score
import numpy as np
import math
import torch
from tqdm import tqdm
import pandas as pd
from tabulate import tabulate
import wandb
from nltk.translate.bleu_score import corpus_bleu

class VQA_Trainer:
    '''Class for Trainer Setup to Train the BART Model for VQA'''

    def __init__(self, model, train_dataloader, eval_dataloader, device, config):
        ''' Constructor '''
        self.model = model
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        self.device = device
        self.tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
        self.scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)  # Initialize ROUGE scorer

        # Semantic similarity model
        self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')  # Pretrained sentence embedding model

        self.optimizer = AdamW(self.model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
        self.scheduler = get_linear_schedule_with_warmup(self.optimizer, num_warmup_steps=config['warmup_steps'], num_training_steps=len(self.train_dataloader) * config['epochs'])

        self.model.config.dropout = config['dropout']
        self.model.config.attention_dropout = config['attention_dropout']
        self.num_beams = config['num_beams']

        wandb.init(project=config['project_name'], config=config)
        wandb.watch(self.model, log="all")

    def evaluate(self):
        ''' For Evaluation at the End of Each Epoch '''
        self.model.eval()
        total_loss = 0
        predictions = []
        references = []
        token_level_accuracies = []
        exact_match_accuracies = []
        semantic_similarities = []
        # meteor_scores = []

        progress_bar = tqdm(self.eval_dataloader, desc="Evaluating")

        for batch in progress_bar:
            with torch.no_grad():
                inputs = {key: val.to(self.device) for key, val in batch.items()}
                outputs = self.model(**inputs)
                total_loss += outputs.loss.item()

                summary_ids = self.model.generate(inputs['input_ids'], max_length=80, num_beams=self.num_beams, early_stopping=True)
                decoded_preds = self.tokenizer.batch_decode(summary_ids, skip_special_tokens=True)

                labels = batch['labels']
                labels = torch.where(labels != -100, labels, self.tokenizer.pad_token_id)
                decoded_refs = self.tokenizer.batch_decode(labels, skip_special_tokens=True)

                predictions.extend([pred.split() for pred in decoded_preds])
                references.extend([[ref.split()] for ref in decoded_refs])

                for pred, ref in zip(decoded_preds, decoded_refs):
                    pred_tokens = pred.split()
                    ref_tokens = ref.split()

                    # Token-level accuracy
                    token_accuracy = sum(1 for p, r in zip(pred_tokens, ref_tokens) if p == r) / max(len(ref_tokens), 1)
                    token_level_accuracies.append(token_accuracy)

                    # Exact Match Accuracy
                    exact_match_accuracy = 1 if pred.strip() == ref.strip() else 0
                    exact_match_accuracies.append(exact_match_accuracy)

                    # Semantic Similarity
                    pred_embedding = self.semantic_model.encode(pred)
                    ref_embedding = self.semantic_model.encode(ref)
                    similarity = cosine_similarity([pred_embedding], [ref_embedding])[0][0]
                    semantic_similarities.append(similarity)

                     # METEOR Score (use tokenized inputs)
                    # meteor = meteor_score([ref_tokens], pred_tokens)  # Pass tokenized lists
                    # meteor_scores.append(meteor)

        avg_loss = total_loss / len(self.eval_dataloader)
        perplexity = math.exp(avg_loss)

        bleu_score = corpus_bleu(references, predictions)

        rouge_1_f1_scores = []
        rouge_2_f1_scores = []
        rouge_l_f1_scores = []
        for pred, ref in zip(decoded_preds, decoded_refs):
            rouge_1 = self.scorer.score(ref, pred)['rouge1']
            rouge_2 = self.scorer.score(ref, pred)['rouge2']
            rouge_l = self.scorer.score(ref, pred)['rougeL']

            rouge_1_f1_scores.append(rouge_1.fmeasure)
            rouge_2_f1_scores.append(rouge_2.fmeasure)
            rouge_l_f1_scores.append(rouge_l.fmeasure)

        avg_token_level_accuracy = np.mean(token_level_accuracies)
        avg_exact_match_accuracy = np.mean(exact_match_accuracies)
        avg_semantic_similarity = np.mean(semantic_similarities)
        # avg_meteor = np.mean(meteor_scores)

        avg_rouge_1_f1 = np.mean(rouge_1_f1_scores)
        avg_rouge_2_f1 = np.mean(rouge_2_f1_scores)
        avg_rouge_l_f1 = np.mean(rouge_l_f1_scores)

        df = pd.DataFrame({
            "Training Loss": [avg_loss],
            "ROUGE-1 (F1)": [avg_rouge_1_f1],
            "ROUGE-2 (F1)": [avg_rouge_2_f1],
            "ROUGE-L (F1)": [avg_rouge_l_f1],
            "Corpus BLEU": [bleu_score],
            "Semantic Similarity": [avg_semantic_similarity],
            # "METEOR": [avg_meteor],
            "Token-Level Accuracy": [avg_token_level_accuracy],
            "Exact Match Accuracy": [avg_exact_match_accuracy],
            "Perplexity": [perplexity]
        })

        print(tabulate(df, headers="keys", tablefmt="psql"))

        metrics = {
            "Validation Loss": avg_loss,
            "perplexity": perplexity,
            "bleu": bleu_score,
            "rouge_1_f1": avg_rouge_1_f1,
            "rouge_2_f1": avg_rouge_2_f1,
            "rouge_l_f1": avg_rouge_l_f1,
            "semantic_similarity": avg_semantic_similarity,
            # "meteor": avg_meteor,
            "token_level_accuracy": avg_token_level_accuracy,
            "exact_match_accuracy": avg_exact_match_accuracy,
        }

        wandb.log(metrics)

        return metrics

    def train_epoch(self):
        ''' To Train for Single Epoch '''
        self.model.train()
        for batch in tqdm(self.train_dataloader, desc="Training"):
            self.optimizer.zero_grad()
            inputs = {key: val.to(self.device) for key, val in batch.items()}
            outputs = self.model(**inputs)
            loss = outputs.loss
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            wandb.log({"train_loss": loss.item()})

    def train(self, epochs):
        ''' To Train for N Number of Epochs Passed from User '''
        for epoch in range(epochs):
            print(f'Epoch {epoch+1}/{epochs}')
            self.train_epoch()
            metrics = self.evaluate()
            print(f"Metrics: {metrics}")
            torch.save(self.model.state_dict(), f"model_{epoch}.pth")

In [15]:
api_key = input("Enter your W&B API key: ")
!wandb login --relogin $api_key

StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:
# config = {
#      "epochs" : 10,
#      "model_name": "facebook/bart-large",
#      "project_name": "VQA_BART_RAG_second_DS",
#     'learning_rate': 9.071880672175604e-05,
#     'weight_decay': 1.0818912251075672e-05,
#     'dropout': 0.4153380025576099,
#     'attention_dropout': 0.22747699513739159,
#     'num_beams': 4,
#     'batch_size': 8,
#     'warmup_steps': 124
# }

In [ ]:
def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log='True')
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log='True')
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    attention_dropout = trial.suggest_float('attention_dropout', 0.1, 0.5)
    num_beams = trial.suggest_int('num_beams', 1, 10)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    warmup_steps = trial.suggest_int('warmup_steps', 0, 1000)

    print(f"Trial parameters: "
          f"learning_rate={learning_rate}, "
          f"weight_decay={weight_decay}, "
          f"dropout={dropout}, "
          f"attention_dropout={attention_dropout}, "
          f"num_beams={num_beams}, "
          f"batch_size={batch_size}, "
          f"warmup_steps={warmup_steps}")


    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BartForConditionalGeneration.from_pretrained('facebook/bart-base').to(device)
    config = {'project_name': 'VQA_BART_Tuning',
        'epochs': 10,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'dropout': dropout,
        'attention_dropout': attention_dropout,
        'num_beams': num_beams,
        'warmup_steps': warmup_steps,
        }

    trainer = VQA_Trainer(model, train_dataloader, val_dataloader, device=device, config=config)

    trainer.train(config['epochs'])

    metrics = trainer.evaluate()
    return metrics['token_level_accuracy']

In [ ]:
!pip install optuna_dashboard

In [ ]:
from optuna_dashboard import run_server
import optuna

storage = optuna.storages.InMemoryStorage()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=4)

In [ ]:
print('Best hyperparameters:', study.best_params)
print('Best Accuracy:', study.best_value)

In [ ]:
import optuna.visualization as vis
from matplotlib import pyplot as plt

fig = vis.plot_optimization_history(study)
fig.show()
plt.savefig("Optimization-history.png")

In [ ]:
fig = vis.plot_parallel_coordinate(study)
fig.show()

plt.savefig('parallel-coordinates.png')

In [ ]:
fig = vis.plot_param_importances(study)
fig.show()

plt.savefig('hyperparameter-importances.png')

In [ ]:
fig = vis.plot_slice(study, params=['learning_rate'])
fig.show()

plt.savefig('learning-rate.png')

In [ ]:
# train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
# val_dataloader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=True)
# test_dataloader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=True)

In [ ]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = BartForConditionalGeneration.from_pretrained(config['model_name']).to(device)

In [ ]:
# trainer = VQA_Trainer(model, train_dataloader, val_dataloader, device=device, config=config)

In [ ]:
# trainer.train(config['epochs'])

In [ ]:
wandb.finish()

In [ ]:
# Inference code

# Load the tokenizer and model (make sure these paths match your setup)
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large')
checkpoint_path = '/kaggle/working/model_9.pth'

# Load your trained model checkpoint
def load_checkpoint(model, file_path):
    checkpoint = torch.load(file_path, map_location=device)
    print('checkpoint keys: ', checkpoint.keys())
    model.load_state_dict(checkpoint)
    print(f"Checkpoint loaded from {file_path}")

load_checkpoint(model, checkpoint_path)
model.eval()

In [ ]:
# Function to generate answers larged on input questions
def ask_question(question, object_name, context,max_length=80):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    combined_input = f"{object_name} {question} {context}"

    inputs = tokenizer(
        combined_input,
        return_tensors="pt",
        max_length=200,
        truncation=True
    ).to(device)

    answer_ids = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    answer = tokenizer.decode(answer_ids[0], skip_special_tokens=True)
    return answer

In [ ]:
test_df = pd.read_csv('/kaggle/working/test_df.csv')
test_df.head()

In [ ]:
import pandas as pd

# Retrieve all the necessary columns from the DataFrame
sampled_questions = test_df['Question']
sampled_objects = test_df['Object']
sampled_context = test_df['Context']
sampled_original_answers = test_df['Answer']  # Original answers

answers = []
for question, object_name, context, original_answer in zip(sampled_questions, sampled_objects, sampled_context, sampled_original_answers):
    generated_answer = ask_question(question, object_name, context)
    answers.append({
        "Question": question,
        "Original Answer": original_answer,
        "Generated Answer": generated_answer
    })

# Convert the answers list into a DataFrame
answers_df = pd.DataFrame(answers)

# Optionally, you can save this DataFrame to a CSV file
answers_df.to_csv('generated_answers.csv', index=False)

# Print the DataFrame to verify
print(answers_df)